In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # whatever is free
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
# os.environ["JAX_LOG_COMPILES"] = "1" # to see jit compiliation triggers

from loguru import logger
import jax
logger.info(f'JAX is using: {jax.devices()}')

2026-05-08 17:28:55.035 | INFO     | __main__:<module>:8 - JAX is using: [CudaDevice(id=0)]


In [4]:
from pathlib import Path

from hydra import compose, initialize
from omegaconf import OmegaConf

from kinetix.environment import ActionType, LogWrapper, ObservationType
from kinetix.environment.env import make_kinetix_env
from kinetix.environment.ued.ued import make_reset_fn_from_config
from kinetix.util import generate_params_from_config

import jax.numpy as jnp
from flax.training.train_state import TrainState
from kinetix.models import ScannedRNN, make_network_from_config
from flax.serialization import to_state_dict
from typing import NamedTuple, Any
from tqdm.auto import tqdm
import numpy as np
import jax.random as jr
from functools import partial

from kinetix.render.renderer_pixels import make_render_pixels_rl

from utils import load_config, make_env_from_config, init_random_policy, make_network, init_network_from_params, make_ppo_step, load_and_replicate_levels

In [8]:
from pathlib import Path
Path('../third_party/kinetix/configs').exists()

True

In [11]:
config = load_config(config_path='../third_party/kinetix/configs',config_name='naive_repo')
rng = jax.random.PRNGKey(config.get('seed',0))

In [12]:
import wandb

wandb_run = wandb.init(
    project="Naive Skill Repo", 
    config=dict(config)
)

wandb.define_metric("repo_step")
wandb.define_metric("pick_rate/*", step_metric="repo_step")
wandb.define_metric("solve_rate/*", step_metric="repo_step")
wandb.define_metric("ep_length/*", step_metric="repo_step")
wandb.define_metric("ep_return/*", step_metric="repo_step")

from pathlib import Path
output_path = Path('outputs') / wandb_run.id
output_path.mkdir(parents=True, exist_ok=True)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sobhannili (sobhannili-epfl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
env, env_params, static_env_params = make_env_from_config(config)
pixel_renderer = make_render_pixels_rl(env_params, static_env_params)
network = make_network(config,env,env_params,static_env_params)

In [ ]:
def wrap_states_with_env(env, env_states, env_params):
    """
    Wrap raw env states through the environment, handling arbitrary batch dimensions.
    
    Args:
        env: The wrapped environment
        env_states: Raw env states (can be 1D, 2D, or higher dimensional)
        env_params: Environment parameters
    
    Returns:
        wrapped_states: States with LogWrapper structure, same shape as input
    """
    # Store original shape
    orig_shape = jax.tree_util.tree_leaves(env_states)[0].shape
    
    # Flatten to 1D for uniform processing
    flat_states = jax.tree.map(
        lambda x: x.reshape((-1,) + x.shape[len(orig_shape):]),
        env_states
    )
    
    flat_size = jax.tree_util.tree_leaves(flat_states)[0].shape[0]
    dummy_rngs = jax.random.split(jr.PRNGKey(0), flat_size)
    
    _, wrapped_flat = jax.vmap(env.reset, in_axes=(0, None, 0))(
        dummy_rngs, env_params, flat_states
    )
    
    # Reshape back to original dimensions
    wrapped_states = jax.tree.map(
        lambda x: x.reshape(orig_shape + x.shape[1:]),
        wrapped_flat
    )
    
    return wrapped_states

In [ ]:
# unique_reset_states = load_and_replicate_levels(
#     config["train_levels_list"],
#     int(config["num_env_copies"]),
# )

# # only for sake of wrapping env_state
# dummy_rngs = jax.random.split(jax.random.PRNGKey(0), len(config["train_levels_list"]) * int(config["num_env_copies"]))
# dummy_rngs = dummy_rngs.reshape(len(config["train_levels_list"]), int(config["num_env_copies"]), -1)

# _, reset_states = jax.vmap(
#     jax.vmap(env.reset, in_axes=(0, None, 0)),
#     in_axes=(0, None, 0),
# )(dummy_rngs, env_params, unique_reset_states)

# # dummy_rngs = jax.random.split(jax.random.PRNGKey(0), len(config["train_levels_list"]))
# # unique_reset_states = jax.vmap(env.reset, in_axes=(0, None, 0))(dummy_rngs, env_params, unique_reset_states)

# config["num_train_envs"] = len(config["train_levels_list"])

Created static_env_params=StaticEnvParams(num_polygons=5, num_circles=2, num_joints=1, num_thrusters=1, max_polygon_vertices=4, num_solver_iterations=10, solver_batch_size=16, do_warm_starting=True, num_static_fixated_polys=4, screen_dim=(500, 500), downscale=4, frame_skip=2, max_shape_size=2, num_motor_bindings=4, num_thruster_bindings=2) when loading evaluation levels.


TypeError: Cannot concatenate arrays with shapes that differ in dimensions other than the one being concatenated: concatenating along dimension 2 for shapes (2, 5, 1, 1), (2, 5, 1, 1), (2, 5, 1, 2), (2, 5, 1, 2), (2, 5, 1, 1), (2, 5, 1, 1).

In [7]:
# TODO:
# random actions baseline per env (for the same bumber of steps) -> per env get same stats
# option to load from the checkpoint path

In [33]:
from typing import NamedTuple, TYPE_CHECKING
if TYPE_CHECKING:
    from kinetix.models.transformer_model import ActorCriticTransformer
    from flax.training.train_state import TrainState

class PolicyState(NamedTuple): # supposed to be vmapped. Skill index is same as the batch index
    train_state: TrainState # includes params
    hstate: jax.Array # sake of compatibility

In [34]:
def get_pytree_element_by_index(pytree, i):
    return jax.tree.map(lambda x: x[i], pytree)

def get_pytree_batch_size(pytree):
    leaf = jax.tree_util.tree_leaves(pytree)[0]
    return leaf.shape[0]

from itertools import chain
def flatten_python_2d_nested_list(nested_list):
    return list(chain.from_iterable(nested_list))

In [35]:
class PolicyMatcher:
    n_skills: int
    n_distinct_envs: int
    skill_repo: PolicyState

    def __init__(self,config,network,env,env_params,rng) -> None:
        self.n_skills = int(config['n_skills'])
        self.n_distinct_envs = int(config['num_train_envs'])
        self.env = env
        self.env_params = env_params
        self.network = network
        
        network_rngs = jr.split(rng, self.n_skills)
        
        def make_init_random_policy(rng):
            return init_random_policy(config, network, env, env_params,rng)
        
        train_state, hstate = jax.vmap(make_init_random_policy)(network_rngs)

        assert get_pytree_batch_size(train_state) == self.n_skills
        self.skill_repo = PolicyState(train_state,hstate)

        self.pixel_renderer = make_render_pixels_rl(env_params, static_env_params)


    def match(self,config,reset_states,rng):
        assert get_pytree_batch_size(reset_states) == self.n_distinct_envs

        env_state = reset_states
        # obs = jax.vmap(env.get_obs)(env_state.env_state) # (INDEP_ENVS_COUNT, )

        initial_pixel_obs = jax.vmap(lambda s: self.pixel_renderer(s.env_state))(env_state).image # (INDEP_ENVS_COUNT, 125, 125, 3)

        policy_indices = jr.choice(rng,self.n_skills,shape=(self.n_distinct_envs,),replace=True)

        selected_policy_params_per_env = jax.tree.map(lambda leaf: leaf[policy_indices], self.skill_repo.train_state.params)
        init_from_params = partial(init_network_from_params, config, self.network)
        train_state, hstate = jax.vmap(init_from_params, in_axes=(0,))(selected_policy_params_per_env)

        return train_state, hstate, policy_indices

    def update_repo(self):
        pass

In [36]:
def append_trajectory_metrics(traj_info, episode_history):
    done_mask = np.array(traj_info["returned_episode"])
    returns   = np.array(traj_info["returned_episode_returns"])
    lengths   = np.array(traj_info["returned_episode_lengths"])
    timesteps = np.array(traj_info["timestep"])
    solved    = np.array(traj_info["returned_episode_solved"])

    G, T, B = done_mask.shape
    for g in range(G):
        g_done  = done_mask[g].reshape(-1)
        g_ret   = returns[g].reshape(-1)
        g_len   = lengths[g].reshape(-1)
        g_tstep = timesteps[g].reshape(-1)
        g_solved= solved[g].reshape(-1)

        idx = np.where(g_done)[0]
        if idx.size == 0:
            continue

        ep_returns = g_ret[idx]
        ep_lengths = g_len[idx]
        ep_end     = g_tstep[idx]
        ep_start   = ep_end - ep_lengths
        ep_solved  = g_solved[idx]

        episode_history["return"][g].extend(ep_returns.tolist())
        episode_history["length"][g].extend(ep_lengths.tolist())
        episode_history["start_step"][g].extend(ep_start.tolist())
        episode_history["end_step"][g].extend(ep_end.tolist())
        episode_history.setdefault("solved", [list() for _ in range(G)])[g].extend(ep_solved.tolist())

    return episode_history

In [37]:
def log_repo_step2wandb(repo_step_perf_history: dict,repo_step:int):
    skill_use_count = {skill_id:len(next(iter(repo_step_perf_history[skill_id].values()))) for skill_id in repo_step_perf_history.keys()}
    skill_pick_rate = {skill_id: usage_count / sum(skill_use_count.values()) for skill_id,usage_count in skill_use_count.items()}  

    data2log = {}
    for skill_id in repo_step_perf_history.keys():
        mean_ep_return = np.mean(flatten_python_2d_nested_list(repo_step_perf_history[skill_id]['return']))
        mean_ep_length = np.mean(flatten_python_2d_nested_list(repo_step_perf_history[skill_id]['length']))
        mean_ep_solved = np.mean(flatten_python_2d_nested_list(repo_step_perf_history[skill_id]['solved']))
        pick_rate = skill_pick_rate[skill_id]

        data2log.update({f'ep_return/policy_{skill_id}':mean_ep_return,
                   f'ep_length/policy_{skill_id}':mean_ep_length,
                   f'solve_rate/policy_{skill_id}':mean_ep_solved,
                   f'pick_rate/policy_{skill_id}':pick_rate})
    

    data2log.update({'repo_step': repo_step})
    wandb.log(data2log)

In [38]:
def group_env_metrics_by_policy(per_env_perf_history,policy_indices):
    policy_indices = np.array(policy_indices)
    per_policy_perf_history = {}

    unique_policy_ids = np.unique(np.array(policy_indices)).tolist()

    for p_id in unique_policy_ids:
        per_policy_perf_history[p_id] = {}
        corresponding_envs_mask = (policy_indices == p_id)
        for key in per_env_perf_history.keys():
            per_policy_perf_history[p_id][key] = np.array(per_env_perf_history[key],dtype=object)[corresponding_envs_mask].tolist()

    return per_policy_perf_history

In [15]:
# from functools import partial

# def make_random_action_run(env, env_params, action_space, G, E):
#     @partial(jax.jit, static_argnames=("num_steps",))
#     def run_random_steps(env_state, rng, num_steps):
#         def env_step(carry, _):
#             env_state, rng = carry
#             rng, action_rng = jr.split(rng)
#             action_rngs = jr.split(action_rng, G * E).reshape(G, E, -1)

#             actions = jax.vmap(jax.vmap(lambda r: action_space.sample(r)))(action_rngs)

#             _, next_env_state, _, _, info = jax.vmap(
#                 jax.vmap(env.step, in_axes=(0, 0, 0, None)),
#                 in_axes=(0, 0, 0, None),
#             )(action_rngs, env_state, actions, env_params)

#             return (next_env_state, rng), info

#         (next_env_state, rng), infos = jax.lax.scan(
#             env_step,
#             (env_state, rng),
#             None,
#             length=num_steps,
#         )
#         return next_env_state, rng, infos  # infos: (T, G, E, ...)

#     return run_random_steps

# num_steps = 100
# G = int(config["num_train_envs"])
# E = int(config["num_env_copies"])

# # reset outside (your preferred way)
# _, env_state = jax.vmap(
#     jax.vmap(env.reset, in_axes=(0, None)),
#     in_axes=(0, None),
# )(env_reset_rngs, env_params)

# action_space = env.action_space(env_params)
# random_action_run = make_random_action_run(env, env_params, action_space, G, E)

# env_state, rng, infos = random_action_run(env_state, rng, num_steps=num_steps)

# # same logging path as PPO
# infos = jax.tree.map(lambda x: jnp.transpose(x, (1, 0, 2)), infos)  # (G, T, E, ...)
# per_env_random_perf_history = {
#     k: [[] for _ in range(G)]
#     for k in ("length", "return", "start_step", "end_step", "solved")
# }

# per_env_random_perf_history = append_trajectory_metrics(infos, per_env_random_perf_history)

In [39]:
def get_random_envs(indep_envs_count, envs_in_batch, env, env_params, rng):
    # One RNG per distinct env
    group_reset_rngs = jr.split(rng, indep_envs_count) # (indep_envs_count,) shared reset rng for each group of envs

    # Replicate within each group
    env_reset_rngs = jnp.repeat(group_reset_rngs[:, None, :], envs_in_batch, axis=1)   # (indep_envs_count, envs_in_batch), where the rng is the same within a group

    obs0, fixed_reset_states = jax.vmap(
        jax.vmap(env.reset, in_axes=(0, None)),
        in_axes=(0, None),
    )(env_reset_rngs, env_params)

    return fixed_reset_states

rng, sub_rng = jr.split(rng)
reset_states = get_random_envs(config["num_train_envs"], config["num_env_copies"], env, env_params, sub_rng)
unique_reset_states = jax.tree.map(lambda x: x[:, 0], reset_states)

In [44]:
INDEP_ENVS_COUNT = int(config['num_train_envs'])
NUM_REPO_UPDATES = int(config['num_repo_updates'])
NUM_PPO_UPDATES = int(config['num_updates'])

rng, sub_rng = jr.split(rng)
policy_matcher = PolicyMatcher(config,network,env,env_params,sub_rng)

ppo_step = make_ppo_step(config, env, env_params)

repo_full_perf_history = []
for repo_update_idx in tqdm(range(NUM_REPO_UPDATES),desc='Policy Repo Updates'): 
    per_env_perf_history = {
        k: [[] for _ in range(INDEP_ENVS_COUNT)]
        for k in ("length", "return", "start_step", "end_step", "solved")
    }
    
    rng, sub_rng = jr.split(rng)
    train_state, hstate, policy_indices = policy_matcher.match(config,unique_reset_states,sub_rng)

    # set to the initial reset state again (in each group)
    env_state = reset_states
    obs = jax.vmap(jax.vmap(env.get_obs))(env_state.env_state)

    for update in tqdm(range(NUM_PPO_UPDATES), desc='PPO Updates', leave=False):
        rng, rollout_rng, update_rng = jax.random.split(rng, 3)
        rollout_rngs = jax.random.split(rollout_rng, INDEP_ENVS_COUNT)
        update_rngs  = jax.random.split(update_rng, INDEP_ENVS_COUNT)

        train_state, hstate, obs, env_state, losses, info = ppo_step(
            train_state, hstate, obs, env_state, rollout_rngs, update_rngs, reset_states
        )
                
        per_env_perf_history = append_trajectory_metrics(info,per_env_perf_history)

    repo_step_perf_history = group_env_metrics_by_policy(per_env_perf_history,policy_indices) # list with (n_policies, )
    log_repo_step2wandb(repo_step_perf_history,repo_update_idx)

    repo_full_perf_history.append(per_env_perf_history)

Policy Repo Updates:   0%|          | 0/20 [00:00<?, ?it/s]

PPO Updates:   0%|          | 0/200 [00:00<?, ?it/s]

TypeError: cannot reshape array of shape (5, 5) (size 25) into shape [5, 2, -1] because the product of specified axis sizes (10) does not evenly divide 25

In [ ]:
import pickle

with open(output_path / 'skill_repo_perf.pkl', 'wb') as file:
    pickle.dump(repo_full_perf_history, file)

# with open(output_path / 'skill_repo_perf.pkl', 'rb') as file:
#     repo_full_perf_history = pickle.load(file)

In [ ]:
# for the first n and last n steps log the mean episode length seperately

Priority List:
- Implementing the random actions variant -> generate the actions before hand based on rngs for all steps. Go similarly on batches on envs. Store 

In [ ]:
# track the skill ids even after they expire (incremental skill ids. THe step that they start from is the indicator)

# metric of skill usefulness: 
#   1. good returns (can be compared to a random policy that is evaluated once per all the envs) -> a multiple (like 1.2 more return compared to the random policy leads to the skill being chosen)
#   2. being a bit better than the frozen state of policy that we started from 

# initlizaing from a pretrained network rather than random policy (for everyone, even the non-skill based) can help with naunce of transformer being sample heavy to learn during short skill evaluation period
# pruning: the one with losest usefulness in a previous window (usefullness also accounts for being picked to. So only getting picked doens't mean you are good, and this accounts for that)

# the mother skill should also get attributed (in terms of skill usability of its children) maybe in a decaying rate over time 